# ==============================================================================
# 🎗️ ONCOLOGY DIAGNOSTICS: BREAST CANCER DETECTION VIA SUPPORT VECTOR CLASSIFIER
# ==============================================================================
### Mathematical Formulation & Theoretical Context:
Support Vector Machines find an optimal separating hyperplane $\mathbf{w}^T \mathbf{x} + b = 0$ that maximizes the geometric margin:
$$M = \frac{2}{\|\mathbf{w}\|}$$

In soft-margin classification, slack variables $\xi_i \ge 0$ allow misclassifications on noisy margins, regulated by penalty budget $C$:
$$\min_{\mathbf{w}, b, \boldsymbol{\xi}} \frac{1}{2} \|\mathbf{w}\|^2 + C \sum_{i=1}^N \xi_i \quad \text{s.t. } y_i(\mathbf{w}^T \phi(\mathbf{x}_i) + b) \ge 1 - \xi_i$$

In non-linear manifolds, Mercer's RBF kernel lifts features into an infinite-dimensional Hilbert space:
$$K(\mathbf{x}_i, \mathbf{x}_j) = \exp(-\gamma \|\mathbf{x}_i - \mathbf{x}_j\|^2)$$


In [1]:
# STEP 1: ENVIRONMENT SETUP
import os
import time
import joblib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split, StratifiedKFold, GridSearchCV
from sklearn.svm import SVC
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.metrics import classification_report, confusion_matrix, roc_auc_score, accuracy_score, recall_score

np.random.seed(42)
plt.style.use('seaborn-v0_8-whitegrid' if 'seaborn-v0_8-whitegrid' in plt.style.available else 'default')
print("✅ STEP 1: Diagnostic Oncology Environment ready.")


✅ STEP 1: Diagnostic Oncology Environment ready.


## 📥 STEP 2 & 3: DATA INGESTION & 3-SECOND SANITY AUDIT
Ingesting the UCI Breast Cancer Wisconsin (Diagnostic) Dataset.


In [2]:
# STEP 2 & 3: LOAD & AUDIT
data_path = 'data/breast_cancer/data.csv'
df = pd.read_csv(data_path)

# Drop redundant id and trailing empty column if present
if 'Unnamed: 32' in df.columns:
    df = df.drop(columns=['Unnamed: 32'])
if 'id' in df.columns:
    df = df.drop(columns=['id'])

df.columns = [c.strip().lower() for c in df.columns]
print(f"📊 Dataset Shape: {df.shape[0]} patients, {df.shape[1]} diagnostic parameters")
print(f"Target Distribution:\n{df['diagnosis'].value_counts(normalize=True).round(4) * 100}%")
df.head(3)


📊 Dataset Shape: 569 patients, 31 diagnostic parameters
Target Distribution:
diagnosis
B    62.74
M    37.26
Name: proportion, dtype: float64%


,diagnosis,radius_mean,texture_mean,perimeter_mean,area_mean,smoothness_mean,compactness_mean,concavity_mean,concave points_mean,symmetry_mean,...,radius_worst,texture_worst,perimeter_worst,area_worst,smoothness_worst,compactness_worst,concavity_worst,concave points_worst,symmetry_worst,fractal_dimension_worst
0,M,17.99,10.38,122.8,1001.0,0.11840,0.27760,0.3001,0.14710,0.2419,...,25.38,17.33,184.6,2019.0,0.1622,0.6656,0.7119,0.2654,0.4601,0.11890
1,M,20.57,17.77,132.9,1326.0,0.08474,0.07864,0.0869,0.07017,0.1812,...,24.99,23.41,158.8,1956.0,0.1238,0.1866,0.2416,0.1860,0.2750,0.08902
2,M,19.69,21.25,130.0,1203.0,0.10960,0.15990,0.1974,0.12790,0.2069,...,23.57,25.53,152.5,1709.0,0.1444,0.4245,0.4504,0.2430,0.3613,0.08758


## 🧹 STEP 4 & 5: SEGREGATION & HYGIENE
Separating features and target `diagnosis` ('M' = Malignant / 1, 'B' = Benign / 0).


In [3]:
# STEP 4 & 5: SEGREGATION
X = df.drop(columns=['diagnosis'])
y = (df['diagnosis'].astype(str).str.strip().str.upper() == 'M').astype(int)

print(f"Features: {X.shape[1]} quantitative cell nuclear attributes")
print(f"Class counts: Benign={sum(y==0)}, Malignant={sum(y==1)}")
print(f"Missing attributes: {X.isnull().sum().sum()}")


Features: 30 quantitative cell nuclear attributes
Class counts: Benign=357, Malignant=212
Missing attributes: 0


## 🔒 STEP 6: LEAK-FREE TRAIN-TEST SPLIT
80/20 stratified split to preserve malignant cancer prevalence.


In [4]:
# STEP 6: STRATIFIED SPLIT
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, random_state=42, stratify=y
)
print(f"Train Cohort: {X_train.shape[0]} patients ({y_train.sum()} Malignant)")
print(f"Test Cohort : {X_test.shape[0]} patients ({y_test.sum()} Malignant)")


Train Cohort: 455 patients (170 Malignant)
Test Cohort : 114 patients (42 Malignant)


## ⚠️ STEP 7: PROOF OF THE MANDATORY SCALING RULE
Demonstrating how unscaled data cripples SVM convergence and accuracy.


In [5]:
# STEP 7: UNSCALED VS SCALED DUEL
# 1. Unscaled SVC
svc_unscaled = SVC(kernel='rbf', random_state=42)
svc_unscaled.fit(X_train, y_train)
acc_unscaled = accuracy_score(y_test, svc_unscaled.predict(X_test))

# 2. Scaled SVC
svc_scaled = Pipeline([
    ('scaler', StandardScaler()),
    ('svc', SVC(kernel='rbf', random_state=42))
])
svc_scaled.fit(X_train, y_train)
acc_scaled = accuracy_score(y_test, svc_scaled.predict(X_test))

print("="*60)
print(f"❌ Unscaled SVM Test Accuracy : {acc_unscaled * 100:.2f}%")
print(f"✅ StandardScaler SVM Accuracy: {acc_scaled * 100:.2f}%")
print(f"📈 Generalization Leap        : +{(acc_scaled - acc_unscaled) * 100:.2f}% Accuracy gain!")
print("="*60)


❌ Unscaled SVM Test Accuracy : 90.35%
✅ StandardScaler SVM Accuracy: 97.37%
📈 Generalization Leap        : +7.02% Accuracy gain!


## ⚔️ STEP 8: KERNEL DUEL (Linear vs RBF vs Polynomial)
Benchmarking separating geometry across different Hilbert spaces.


In [6]:
# STEP 8: KERNEL BENCHMARK
kernels = ['linear', 'rbf', 'poly']
kernel_results = {}

for k in kernels:
    pipe = Pipeline([
        ('scaler', StandardScaler()),
        ('svc', SVC(kernel=k, random_state=42))
    ])
    pipe.fit(X_train, y_train)
    preds = pipe.predict(X_test)
    kernel_results[k] = {
        'Accuracy': accuracy_score(y_test, preds),
        'Recall': recall_score(y_test, preds),
        'Support_Vectors': pipe.named_steps['svc'].n_support_.sum()
    }

print("🏆 KERNEL BENCHMARK RESULTS:")
for k, res in kernel_results.items():
    print(f"   Kernel '{k:6s}' -> Acc: {res['Accuracy']*100:.2f}% | Recall: {res['Recall']*100:.2f}% | Active SVs: {res['Support_Vectors']}/{len(X_train)}")


🏆 KERNEL BENCHMARK RESULTS:
   Kernel 'linear' -> Acc: 96.49% | Recall: 90.48% | Active SVs: 38/455
   Kernel 'rbf   ' -> Acc: 97.37% | Recall: 92.86% | Active SVs: 107/455
   Kernel 'poly  ' -> Acc: 88.60% | Recall: 69.05% | Active SVs: 149/455


## 🎯 STEP 9: JOINT HYPERPARAMETER TUNING ($C$ and $\gamma$) VIA 5-FOLD CV
Optimizing margin penalty $C$ and Gaussian radius $\gamma$ simultaneously on a log grid.


In [7]:
# STEP 9: 2D GRID SEARCH
param_grid = {
    'svc__C': [0.1, 1, 10, 100],
    'svc__gamma': ['scale', 0.001, 0.01, 0.1, 1]
}

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
pipe_tune = Pipeline([
    ('scaler', StandardScaler()),
    ('svc', SVC(kernel='rbf', probability=True, random_state=42))
])

grid = GridSearchCV(pipe_tune, param_grid=param_grid, cv=cv, scoring='recall', n_jobs=-1)
grid.fit(X_train, y_train)

best_model = grid.best_estimator_
print(f"🏆 Best Oncology Hyperparameters: {grid.best_params_}")
print(f"🎯 Best 5-Fold Cross-Validation Recall: {grid.best_score_ * 100:.2f}%")


/home/python/03_Custom_addons/ML/.venv/lib/python3.11/site-packages/sklearn/svm/_base.py:236: FutureWarning: The `probability` parameter was deprecated in 1.9 and will be removed in version 1.11. Use `CalibratedClassifierCV(SVC(), ensemble=False)` instead of `SVC(probability=True)`
  warnings.warn(
/home/python/03_Custom_addons/ML/.venv/lib/python3.11/site-packages/sklearn/svm/_base.py:236: FutureWarning: The `probability` parameter was deprecated in 1.9 and will be removed in version 1.11. Use `CalibratedClassifierCV(SVC(), ensemble=False)` instead of `SVC(probability=True)`
  warnings.warn(
/home/python/03_Custom_addons/ML/.venv/lib/python3.11/site-packages/sklearn/svm/_base.py:236: FutureWarning: The `probability` parameter was deprecated in 1.9 and will be removed in version 1.11. Use `CalibratedClassifierCV(SVC(), ensemble=False)` instead of `SVC(probability=True)`
  warnings.warn(
/home/python/03_Custom_addons/ML/.venv/lib/python3.11/site-packages/sklearn/svm/_base.py:236: Future

/home/python/03_Custom_addons/ML/.venv/lib/python3.11/site-packages/sklearn/svm/_base.py:236: FutureWarning: The `probability` parameter was deprecated in 1.9 and will be removed in version 1.11. Use `CalibratedClassifierCV(SVC(), ensemble=False)` instead of `SVC(probability=True)`
  warnings.warn(
/home/python/03_Custom_addons/ML/.venv/lib/python3.11/site-packages/sklearn/svm/_base.py:236: FutureWarning: The `probability` parameter was deprecated in 1.9 and will be removed in version 1.11. Use `CalibratedClassifierCV(SVC(), ensemble=False)` instead of `SVC(probability=True)`
  warnings.warn(
/home/python/03_Custom_addons/ML/.venv/lib/python3.11/site-packages/sklearn/svm/_base.py:236: FutureWarning: The `probability` parameter was deprecated in 1.9 and will be removed in version 1.11. Use `CalibratedClassifierCV(SVC(), ensemble=False)` instead of `SVC(probability=True)`
  warnings.warn(
/home/python/03_Custom_addons/ML/.venv/lib/python3.11/site-packages/sklearn/svm/_base.py:236: Future

/home/python/03_Custom_addons/ML/.venv/lib/python3.11/site-packages/sklearn/svm/_base.py:236: FutureWarning: The `probability` parameter was deprecated in 1.9 and will be removed in version 1.11. Use `CalibratedClassifierCV(SVC(), ensemble=False)` instead of `SVC(probability=True)`
  warnings.warn(
/home/python/03_Custom_addons/ML/.venv/lib/python3.11/site-packages/sklearn/svm/_base.py:236: FutureWarning: The `probability` parameter was deprecated in 1.9 and will be removed in version 1.11. Use `CalibratedClassifierCV(SVC(), ensemble=False)` instead of `SVC(probability=True)`
  warnings.warn(
/home/python/03_Custom_addons/ML/.venv/lib/python3.11/site-packages/sklearn/svm/_base.py:236: FutureWarning: The `probability` parameter was deprecated in 1.9 and will be removed in version 1.11. Use `CalibratedClassifierCV(SVC(), ensemble=False)` instead of `SVC(probability=True)`
  warnings.warn(
/home/python/03_Custom_addons/ML/.venv/lib/python3.11/site-packages/sklearn/svm/_base.py:236: Future

/home/python/03_Custom_addons/ML/.venv/lib/python3.11/site-packages/sklearn/svm/_base.py:236: FutureWarning: The `probability` parameter was deprecated in 1.9 and will be removed in version 1.11. Use `CalibratedClassifierCV(SVC(), ensemble=False)` instead of `SVC(probability=True)`
  warnings.warn(
/home/python/03_Custom_addons/ML/.venv/lib/python3.11/site-packages/sklearn/svm/_base.py:236: FutureWarning: The `probability` parameter was deprecated in 1.9 and will be removed in version 1.11. Use `CalibratedClassifierCV(SVC(), ensemble=False)` instead of `SVC(probability=True)`
  warnings.warn(
/home/python/03_Custom_addons/ML/.venv/lib/python3.11/site-packages/sklearn/svm/_base.py:236: FutureWarning: The `probability` parameter was deprecated in 1.9 and will be removed in version 1.11. Use `CalibratedClassifierCV(SVC(), ensemble=False)` instead of `SVC(probability=True)`
  warnings.warn(
/home/python/03_Custom_addons/ML/.venv/lib/python3.11/site-packages/sklearn/svm/_base.py:236: Future

/home/python/03_Custom_addons/ML/.venv/lib/python3.11/site-packages/sklearn/svm/_base.py:236: FutureWarning: The `probability` parameter was deprecated in 1.9 and will be removed in version 1.11. Use `CalibratedClassifierCV(SVC(), ensemble=False)` instead of `SVC(probability=True)`
  warnings.warn(
/home/python/03_Custom_addons/ML/.venv/lib/python3.11/site-packages/sklearn/svm/_base.py:236: FutureWarning: The `probability` parameter was deprecated in 1.9 and will be removed in version 1.11. Use `CalibratedClassifierCV(SVC(), ensemble=False)` instead of `SVC(probability=True)`
  warnings.warn(
/home/python/03_Custom_addons/ML/.venv/lib/python3.11/site-packages/sklearn/svm/_base.py:236: FutureWarning: The `probability` parameter was deprecated in 1.9 and will be removed in version 1.11. Use `CalibratedClassifierCV(SVC(), ensemble=False)` instead of `SVC(probability=True)`
  warnings.warn(
/home/python/03_Custom_addons/ML/.venv/lib/python3.11/site-packages/sklearn/svm/_base.py:236: Future

🏆 Best Oncology Hyperparameters: {'svc__C': 10, 'svc__gamma': 'scale'}
🎯 Best 5-Fold Cross-Validation Recall: 96.47%


## 📊 STEP 10 & 11: CLINICAL EVALUATION & CONFUSION MATRIX
Reviewing diagnostic safety, false negative rates, and ROC-AUC.


In [8]:
# STEP 10 & 11: EVALUATION REPORT
y_pred = best_model.predict(X_test)
y_prob = best_model.predict_proba(X_test)[:, 1]

cm = confusion_matrix(y_test, y_pred)
print("📋 CLINICAL DIAGNOSTIC CLASSIFICATION REPORT:")
print(classification_report(y_test, y_pred, target_names=['Benign', 'Malignant']))

print(f"🎯 Test Accuracy   : {accuracy_score(y_test, y_pred)*100:.2f}%")
print(f"🎯 Malignant Recall: {recall_score(y_test, y_pred)*100:.2f}%")
print(f"🎯 ROC-AUC Score   : {roc_auc_score(y_test, y_prob)*100:.2f}%")

print(f"\nConfusion Matrix:\n{cm}")
tn, fp, fn, tp = cm.ravel()
print(f"False Negatives (Missed Malignancies): {fn} (Safety Goal: 0)")


📋 CLINICAL DIAGNOSTIC CLASSIFICATION REPORT:
              precision    recall  f1-score   support

      Benign       0.96      1.00      0.98        72
   Malignant       1.00      0.93      0.96        42

    accuracy                           0.97       114
   macro avg       0.98      0.96      0.97       114
weighted avg       0.97      0.97      0.97       114

🎯 Test Accuracy   : 97.37%
🎯 Malignant Recall: 92.86%
🎯 ROC-AUC Score   : 99.27%

Confusion Matrix:
[[72  0]
 [ 3 39]]
False Negatives (Missed Malignancies): 3 (Safety Goal: 0)


## 💾 STEP 12 & 13: PRODUCTION SERIALIZATION & PATHOLOGY SMOKE TEST
Deploying the pipeline and testing sub-millisecond bedside pathology inference latency.


In [9]:
# STEP 12 & 13: SERIALIZATION & SMOKE TEST
os.makedirs('production_models', exist_ok=True)
model_path = 'production_models/breast_cancer_svc_pipeline.joblib'
joblib.dump(best_model, model_path, compress=3)
print(f"💾 Production pipeline saved to: {model_path} ({os.path.getsize(model_path):,} bytes)")

loaded_pipe = joblib.load(model_path)
sample_patient = X_test.iloc[[0]]

t0 = time.perf_counter()
pred_class = loaded_pipe.predict(sample_patient)[0]
pred_prob = loaded_pipe.predict_proba(sample_patient)[0, 1]
latency_ms = (time.perf_counter() - t0) * 1000

print(f"\n🔬 LIVE BEDSIDE PATHOLOGY INFERENCE:")
print(f"   Diagnosis Assessment : {'🚨 MALIGNANT' if pred_class == 1 else '🟢 BENIGN'}")
print(f"   Malignancy Likelihood: {pred_prob * 100:.2f}%")
print(f"   Diagnostic Latency   : {latency_ms:.3f} ms (SLA < 2.0ms: {'PASS' if latency_ms < 2.0 else 'CHECK'})")


💾 Production pipeline saved to: production_models/breast_cancer_svc_pipeline.joblib (22,110 bytes)

🔬 LIVE BEDSIDE PATHOLOGY INFERENCE:
   Diagnosis Assessment : 🟢 BENIGN
   Malignancy Likelihood: 0.02%
   Diagnostic Latency   : 7.877 ms (SLA < 2.0ms: CHECK)
